### this notebook is to test the functions in part 2 of the coursework

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/Users/sisigao/Desktop/Birkbeck_master/Natural_language_processing/0_coursework/cw-pack-2026/texts/hansard10000.csv")
display(df.head(2))

,speech,party,constituency,date,speech_class,major_heading,year,speakername
0,We will now suspend for three minutes for sani...,Conservative,Ribble Valley,2021-03-11,Speech,Contingencies Fund (No. 2) Bill,2021,Nigel Evans
1,I am now beginning to share the indignation of...,Labour,City of Chester,2020-11-24,Speech,Exiting the European Union,2020,Christian Matheson


In [3]:
# rename the labours(Co-op) in party column to labour
df["party"] = df["party"].replace("Labour (Co-op)", "Labour")
display(df.head(2))

,speech,party,constituency,date,speech_class,major_heading,year,speakername
0,We will now suspend for three minutes for sani...,Conservative,Ribble Valley,2021-03-11,Speech,Contingencies Fund (No. 2) Bill,2021,Nigel Evans
1,I am now beginning to share the indignation of...,Labour,City of Chester,2020-11-24,Speech,Exiting the European Union,2020,Christian Matheson


In [4]:
# remove any rows where the value of the ‘party’ column is not one of the four
# most common party names, and remove the ‘Speaker’ value.
parties_in_df = df["party"].unique()
# print(parties_in_df)

print(df["party"].value_counts())
df_cleaned = df[df["party"].isin(["Labour", "Conservative", "Scottish National Party", "Liberal Democrat"])]

# remove the speakername column
df_cleaned = df_cleaned.drop(columns=["speakername"])
display(df_cleaned.head(2))
print(len(df_cleaned))

# remove any rows where the value in the ‘speech class’ column is not ‘Speech’.
df_cleaned = df_cleaned[df_cleaned["speech_class"] == "Speech"]
print(len(df_cleaned))

# remove any rows where the text in the ‘speech’ column is less than 1000 characters long.
df_cleaned = df_cleaned[df_cleaned["speech"].str.len() >= 1000]
print(len(df_cleaned))
print(df_cleaned.shape)

party
Conservative                        6192
Labour                              2108
Scottish National Party              560
Liberal Democrat                     221
Speaker                              208
Democratic Unionist Party            140
Independent                           58
Plaid Cymru                           51
Social Democratic & Labour Party      21
Green Party                           17
Alliance                              12
Alba Party                             1
Name: count, dtype: int64


,speech,party,constituency,date,speech_class,major_heading,year
0,We will now suspend for three minutes for sani...,Conservative,Ribble Valley,2021-03-11,Speech,Contingencies Fund (No. 2) Bill,2021
1,I am now beginning to share the indignation of...,Labour,City of Chester,2020-11-24,Speech,Exiting the European Union,2020


9081
9081
2112
(2112, 7)


### the following is to test the code for question 2b

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score

In [6]:
# !pip install scikit-learn

In [7]:
# question 2b
# vectorise using TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words="english", max_features=3000)

# can't vectorise the whole dataset. it will cause data leakage.
speeches = df_cleaned["speech"].values
y = df_cleaned["party"].values

speech_train, speech_test, y_train, y_test = train_test_split(
    speeches, y, test_size=0.2, random_state=26, stratify=y
)

X_train = vectorizer.fit_transform(speech_train)
X_test = vectorizer.transform(speech_test)

In [8]:
# train a Random Forest classifier
rf = RandomForestClassifier(random_state=26, n_estimators=300)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print("macro F1-score:", f1_score(y_test, rf_pred, average="macro"))
print("Random Forest Classification Report:")
print(classification_report(y_test, rf_pred))

macro F1-score: 0.41095466923275725
Random Forest Classification Report:
                         precision    recall  f1-score   support

           Conservative       0.70      0.99      0.82       250
                 Labour       0.82      0.43      0.57       125
       Liberal Democrat       0.00      0.00      0.00        15
Scottish National Party       0.83      0.15      0.26        33

               accuracy                           0.72       423
              macro avg       0.59      0.39      0.41       423
           weighted avg       0.72      0.72      0.67       423



/Users/sisigao/Desktop/Birkbeck_master/Natural_language_processing/0_coursework/cw_venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/sisigao/Desktop/Birkbeck_master/Natural_language_processing/0_coursework/cw_venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/sisigao/Desktop/Birkbeck_master/Natural_language_processing/0_coursework/cw_venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defin

In [9]:
# train a SVM with linear kernel
svm = LinearSVC(random_state=26)
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)
print("macro F1-score:", f1_score(y_test, svm_pred, average="macro"))
print("SVM (Linear) Classification Report:")
print(classification_report(y_test, svm_pred))

macro F1-score: 0.5221296296296296
SVM (Linear) Classification Report:
                         precision    recall  f1-score   support

           Conservative       0.76      0.88      0.82       250
                 Labour       0.65      0.60      0.62       125
       Liberal Democrat       1.00      0.07      0.12        15
Scottish National Party       0.76      0.39      0.52        33

               accuracy                           0.73       423
              macro avg       0.79      0.49      0.52       423
           weighted avg       0.74      0.73      0.71       423



### the following is to test the code of question 2c 

In [11]:
# question 2c with n-grams
vectorizer_ngram = TfidfVectorizer(
    stop_words="english", max_features=3000, ngram_range= (1, 3)
)

# modify code here to avoid data leakage
X_train_ng = vectorizer_ngram.fit_transform(speech_train)
X_test_ng = vectorizer_ngram.transform(speech_test)


# train a Random Forest classifier
rf_ng = RandomForestClassifier(random_state=26, n_estimators=300)
rf_ng.fit(X_train_ng, y_train)
rf_ng_pred = rf_ng.predict(X_test_ng)
print("macro F1-score:", f1_score(y_test, rf_ng_pred, average="macro"))
print("Random Forest Classification Report:")
print(classification_report(y_test, rf_ng_pred))

# train a SVM with linear kernel
svm_ng = LinearSVC(random_state=26)
svm_ng.fit(X_train_ng, y_train)
svm_ng_pred = svm_ng.predict(X_test_ng)
print("macro F1-score:", f1_score(y_test, svm_ng_pred, average="macro"))
print("SVM (Linear) Classification Report:")
print(classification_report(y_test, svm_ng_pred))

macro F1-score: 0.4088757549123403
Random Forest Classification Report:
                         precision    recall  f1-score   support

           Conservative       0.71      0.97      0.82       250
                 Labour       0.71      0.42      0.53       125
       Liberal Democrat       0.00      0.00      0.00        15
Scottish National Party       0.75      0.18      0.29        33

               accuracy                           0.71       423
              macro avg       0.54      0.39      0.41       423
           weighted avg       0.69      0.71      0.66       423

macro F1-score: 0.5339907657277787
SVM (Linear) Classification Report:
                         precision    recall  f1-score   support

           Conservative       0.77      0.89      0.83       250
                 Labour       0.68      0.63      0.65       125
       Liberal Democrat       1.00      0.07      0.12        15
Scottish National Party       0.81      0.39      0.53        33

       

/Users/sisigao/Desktop/Birkbeck_master/Natural_language_processing/0_coursework/cw_venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/sisigao/Desktop/Birkbeck_master/Natural_language_processing/0_coursework/cw_venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/sisigao/Desktop/Birkbeck_master/Natural_language_processing/0_coursework/cw_venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defin

### the following code is to test the code for question 2d

In [12]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords, wordnet
from nltk import pos_tag, word_tokenize

# nltk.download("punkt")
# nltk.download("averaged_perceptron_tagger")
# nltk.download("wordnet")
# nltk.download("stopwords")
# nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger_eng')

In [13]:
# lemmatise to reduce vocabulary noise
# POS filtering to keep only nouns, verbs, adj. most likely 
# keep bigram for party-specific phrases (normally two words)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def get_wordnet_pos(treebank_tag):
    """Convert Penn Treebank POS tags to WordNet POS tags for lemmatizer."""
    if treebank_tag.startswith("J"):
        return wordnet.ADJ
    elif treebank_tag.startswith("V"):
        return wordnet.VERB
    elif treebank_tag.startswith("N"):
        return wordnet.NOUN
    elif treebank_tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # default

In [14]:
def custom_tokenizer(text):
    # 1. lowercase and remove non-alpha characters
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)

    # 2. tokenize
    tokens = word_tokenize(text)

    # 3. POS tag
    tagged = pos_tag(tokens)

    # 4. keep only nouns, verbs, adjectives, adverbs; lemmatize; remove stopwords
    lemmatized = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in tagged
        if word not in stop_words
        and len(word) > 2
        and tag.startswith(("N", "V", "J", "R"))  # nouns, verbs, adj, adv
    ]

    return lemmatized

In [15]:
# vectorise with custom tokenizer + bigrams, max 3000 features
vectorizer_custom = TfidfVectorizer(
    tokenizer=custom_tokenizer,
    max_features=3000,
    # unigrams + bigrams (trigrams add noise with lemmatized tokens)
    ngram_range=(1, 2),   
    # ignore very rare terms (reduces noise)
    min_df=2,             
    # log-scale TF dampens very frequent terms
    sublinear_tf=True,    
)

X_train_c = vectorizer_custom.fit_transform(speech_train)
X_test_c = vectorizer_custom.transform(speech_test)

/Users/sisigao/Desktop/Birkbeck_master/Natural_language_processing/0_coursework/cw_venv/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [18]:
# evaluate classifiers, report the best
classifiers = {
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=26),
    "SVM (Linear)":  LinearSVC(random_state=26),
}

best_name, best_score, best_pred = None, 0, None

for name, clf in classifiers.items():
    clf.fit(X_train_c, y_train)
    pred = clf.predict(X_test_c)
    score = f1_score(y_test, pred, average="macro")
    print(f"{name} — macro F1: {score:.4f}")
    if score > best_score:
        best_score = score
        best_name  = name
        best_pred  = pred

print(f"\nBest classifier: {best_name}")
print(f"macro F1-score: {best_score:.4f}")
print("Classification Report:")
print(classification_report(y_test, best_pred))

Random Forest — macro F1: 0.4268
SVM (Linear) — macro F1: 0.5368

Best classifier: SVM (Linear)
macro F1-score: 0.5368
Classification Report:
                         precision    recall  f1-score   support

           Conservative       0.82      0.90      0.86       250
                 Labour       0.66      0.67      0.67       125
       Liberal Democrat       1.00      0.07      0.12        15
Scottish National Party       0.68      0.39      0.50        33

               accuracy                           0.76       423
              macro avg       0.79      0.51      0.54       423
           weighted avg       0.77      0.76      0.75       423

